# SENTINEL-GNSS — Colab Training Notebook

**Before running:** Runtime ▸ Change runtime type ▸ **T4 GPU**

| What lives where | |
|---|---|
| Code + all data CSVs | GitHub (cloned in Step 1) |
| Feature windows (.npz) | Built fresh each session from CSV |
| Checkpoints + figures | Google Drive (survives disconnects) |

**If Colab disconnects mid-training:** Re-run Steps 1–3, then run the recovery cell at the bottom — it resumes from the last saved checkpoint automatically.

## Step 1 — Clone repo from GitHub
All code and processed CSVs (including the labelled dataset) are pulled directly from GitHub.

In [ ]:
import shutil
shutil.rmtree('/content/sentinel-gnss')

import os; os.chdir('/content')

In [ ]:
import os, shutil

GITHUB_REPO = 'https://github.com/Jorshuare/AI-Based-Prediction-for-GNSS-Signal-Degradation.git'
REPO_DIR    = '/content/sentinel-gnss'

# Always anchor to /content first so the kernel is never left in a deleted directory
%cd /content

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repo already cloned — pulling latest ...')
    %cd {REPO_DIR}
    !git pull
else:
    # Remove stale directory if it exists without .git (e.g. leftover from failed clone)
    if os.path.exists(REPO_DIR):
        print('Removing stale directory ...')
        shutil.rmtree(REPO_DIR)
    print('Cloning repo ...')
    !git clone {GITHUB_REPO} {REPO_DIR}
    %cd {REPO_DIR}

print(f'\nWorking directory: {os.getcwd()}')
!git log --oneline -5

# Confirm labelled CSV came down with the repo
csv_path = f'{REPO_DIR}/data/labelled/sentinel_gnss_labelled.csv'
assert os.path.exists(csv_path), f'CSV not found at {csv_path}'
import pandas as pd
df = pd.read_csv(csv_path)
print(f'\nDataset: {len(df):,} rows × {len(df.columns)} columns — ready.')

## Step 2 — Install extra dependencies + verify GPU
Colab already ships with PyTorch, NumPy, pandas, scikit-learn, matplotlib, seaborn, scipy.  
Only `imbalanced-learn` (SMOTE) needs installing.

In [ ]:
!pip install -q imbalanced-learn

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {props.total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

## Step 3 — Mount Google Drive (checkpoints + figures only)
Checkpoints are saved to Drive so training survives disconnects.  
Data CSVs are **not** needed here — they came from GitHub in Step 1.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

REPO_DIR   = '/content/sentinel-gnss'
DRIVE_BASE = '/content/drive/MyDrive/sentinel-gnss'

# Folders that must persist across sessions
PAIRS = [
    (f'{REPO_DIR}/results/models/checkpoints', f'{DRIVE_BASE}/checkpoints'),
    (f'{REPO_DIR}/results/figures',            f'{DRIVE_BASE}/figures'),
]

for local, on_drive in PAIRS:
    os.makedirs(on_drive, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.isdir(local) and not os.path.islink(local):
        shutil.rmtree(local)   # remove empty local dir
    if not os.path.islink(local):
        os.symlink(on_drive, local)
    print(f'  {os.path.basename(local):15s} → {on_drive}')

# Report any existing checkpoints (from a previous run)
ckpt_dir = f'{DRIVE_BASE}/checkpoints'
ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
if ckpts:
    print(f'\nExisting checkpoints on Drive ({len(ckpts)}):')
    for c in ckpts:
        print(f'  {c}')
    print('\nRun Step 5 with --resume to continue training.')
else:
    print('\nNo existing checkpoints — will start fresh.')

## Step 4 — Build feature windows
Reads `data/labelled/sentinel_gnss_labelled.csv` → produces sliding-window tensors.  
**Skips automatically if windows already exist** (safe to re-run).

In [ ]:
%cd /content/sentinel-gnss
!python -m src.models.feature_prep

# Sanity check
import numpy as np
print('\nWindow shapes:')
for split in ('train', 'val', 'test'):
    d = np.load(f'data/processed/windows/{split}.npz')
    print(f'  {split:5s}  X={d["X"].shape}  labels(5s): CLEAN={np.sum(d["y_5s"]==0):,}  WARNING={np.sum(d["y_5s"]==1):,}  DEGRADED={np.sum(d["y_5s"]==2):,}')

## Step 5 — Train
`--resume` continues from the last Drive checkpoint if it exists, otherwise starts fresh.  
Checkpoints are saved directly to Drive every 10 epochs + whenever a new best val F1 is reached.

In [ ]:
import shutil, os, glob

# Wipe ALL checkpoints from Drive so class-weight change takes full effect
drive_ckpt = '/content/drive/MyDrive/sentinel-gnss/checkpoints'
local_ckpt = '/content/sentinel-gnss/results/models/checkpoints'

for ckpt_path in [drive_ckpt, local_ckpt]:
    if os.path.exists(ckpt_path):
        for f in glob.glob(f'{ckpt_path}/*'):
            os.remove(f)
        print(f'Cleared: {ckpt_path}')

%cd /content/sentinel-gnss
# Fresh training run — class weights [1.0, 2.0, 3.0], batch_size=256 on T4
!python -m src.models.train --batch_size 256

## Step 6 — Evaluate
Loads `checkpoint_best.pt` from Drive, runs all 14 analyses, saves figures to Drive.

In [ ]:
%cd /content/sentinel-gnss
!python -m src.models.evaluate

## Step 7 — View all figures inline

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob('/content/sentinel-gnss/results/figures/*.png'))
print(f'{len(figs)} figures saved to Drive:')
for f in figs:
    name = f.split('/')[-1]
    print(f'  {name}')
    display(Image(filename=f, width=950))

## Step 8 — Baselines (Tier 1–3)

Run trivial, domain-rule, and classical-ML baselines for comparison table.
Takes ~5–10 min on CPU (RF 500 trees, XGBoost 400 estimators).

In [ ]:
import subprocess, sys
# Install XGBoost if not already present
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

# Run all Tier 1–3 baselines (MajorityClass, CNR threshold, RF, XGBoost)
!python -m src.models.baselines

## Step 9 — Ablations (Tier 4)

Train LSTM-only and Transformer-only variants.
Each uses its own checkpoint directory (`checkpoints_lstm_only/`, `checkpoints_transformer_only/`).
Same data, same loss, same hyperparameters — only architecture differs.

In [ ]:
# LSTM-only ablation — no Transformer encoder
# Expected: fewer epochs to converge, lower macro-F1 than full model
!python -m src.models.train --model_type lstm_only --batch_size 256

In [ ]:
# Transformer-only ablation — no LSTM (mean pooling over sequence)
# Expected: lower macro-F1 than full model due to missing sequential state
!python -m src.models.train --model_type transformer_only --batch_size 256

In [ ]:
# Print the full comparison table (includes DL ablation results if available)
!python -m src.models.baselines --include_ablations

---
## Disconnect recovery
If Colab restarted, re-run **Steps 1 → 3** first, then run this cell.  
It finds the latest Drive checkpoint and resumes training exactly from that epoch.

In [ ]:
%cd /content/sentinel-gnss
# Resumes from last Drive checkpoint automatically
!python -m src.models.train --resume --batch_size 256

## 9 — Show figures inline
Browse all saved figures without leaving Colab.

In [ ]:
%cd /content/sentinel-gnss
!python -m src.models.train --resume --batch_size 256